# Process Single ProQuest File

This notebook processes a single ProQuest CSV file without affecting other processed datasets.

**Input:** `processed-data-2010-2025_updated.csv.gz`  
**Output:** `processed_data_2010-2025_updated.jsonl.gz`

In [8]:
# Import required libraries
import pandas as pd
from src.utils import utils
import os

print("Libraries imported successfully!")

Libraries imported successfully!


## Configuration

Set the input and output paths here:

In [9]:
# Input file
INPUT_FILE = "/data/mourad/narratives/proquest/raw_from_TDM/processed-data-2010-2025_updated.csv.gz"

# Reference file containing already-processed sentences
REFERENCE_FILE = "/data/mourad/narratives/proquest/raw_from_TDM/processed-data-2010-2025.csv.gz"

# Output file (won't overwrite existing files)
OUTPUT_FILE = "/data/mourad/narratives/proquest/processed_data_2010-2025_updated.jsonl.gz"

print(f"Input:  {INPUT_FILE}")
print(f"Reference: {REFERENCE_FILE}")
print(f"Output:    {OUTPUT_FILE}")

Input:  /data/mourad/narratives/proquest/raw_from_TDM/processed-data-2010-2025_updated.csv.gz
Reference: /data/mourad/narratives/proquest/raw_from_TDM/processed-data-2010-2025.csv.gz
Output:    /data/mourad/narratives/proquest/processed_data_2010-2025_updated.jsonl.gz


## Load Data

In [10]:
# Load the CSV file
print(f"Loading {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE, compression="gzip", sep="\t")

# Basic data cleaning
total_rows = len(df)
df = df[df.text.notna()].copy()
empty_texts = total_rows - len(df)
non_empty_rows = len(df)

# Split year_month into separate columns
df[['year', 'month']] = df['year_month'].str.split('-', expand=True)
df.year = df.year.astype(int)
df.month = df.month.astype(int)

print(f"\nTotal rows: {total_rows:,}")
print(f"Rows with empty text: {empty_texts:,} ({empty_texts/total_rows:.1%})")
print(f"Dataset size after removing empty text: {len(df):,}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"Date range: {df.year.min()}-{df.year.max()}")

Loading /data/mourad/narratives/proquest/raw_from_TDM/processed-data-2010-2025_updated.csv.gz...

Total rows: 1,746,611
Rows with empty text: 25,425 (1.5%)
Dataset size after removing empty text: 1,721,186

Columns: ['file_id', 'year_month', 'title', 'loc', 'text', 'year', 'month']
Date range: 2010-2025


## Remove Overlapping Sentences

Load the previously processed dataset and drop any rows from the updated file that share identical sentences.


In [11]:
print("Checking for overlapping sentences with reference dataset...")

reference_df = pd.read_csv(
    REFERENCE_FILE,
    compression="gzip",
    sep="\t",
    usecols=["text"],
)
reference_rows = len(reference_df)
reference_texts = set(reference_df["text"].dropna())
del reference_df

print(f"Reference rows: {reference_rows:,}")
print(f"Unique reference texts: {len(reference_texts):,}")

overlap_mask = df["text"].isin(reference_texts)
overlap_rows = int(overlap_mask.sum())
df = df[~overlap_mask].reset_index(drop=True)
del reference_texts

overlap_pct = overlap_rows / non_empty_rows if non_empty_rows else 0
print(f"\nRows overlapping with reference: {overlap_rows:,} ({overlap_pct:.1%})")
print(f"Dataset size after removing overlaps: {len(df):,}")


Checking for overlapping sentences with reference dataset...
Reference rows: 1,263,586
Unique reference texts: 907,699

Rows overlapping with reference: 1,308,961 (76.0%)
Dataset size after removing overlaps: 412,225


## Location Processing Functions

These functions clean and standardize location data from ProQuest.

In [12]:
def extend_substrings(input_list):
    """
    Creates a mapping to extend partial state names to their full names.
    For example, maps 'Mass' to 'Massachusetts' if 'Massachusetts' exists in region_mapping.
    
    Args:
        input_list: List of state names/abbreviations to process
        
    Returns:
        Dictionary mapping partial names to full names
    """
    transformation_map = {}
    filtered_list = [word for word in input_list if word is not None]
    sorted_list = sorted(filtered_list, key=len, reverse=True)

    for word in sorted_list:
        for longer_word in utils.region_mapping.keys():
            if longer_word.startswith(word) and word != longer_word:
                transformation_map[word] = longer_word
                break
        else:
            transformation_map[word] = word

    transformation_map[None] = None
    return transformation_map


def combine_cities(group):
    """
    Combines city entries that appear with different state abbreviations.
    For cities with multiple state entries, fills missing states with the most common one.
    
    Args:
        group: DataFrame group containing rows for a single city
        
    Returns:
        Processed DataFrame group with standardized state names
    """
    unique_states = group.state.unique()
    expand_states = extend_substrings(unique_states)
    
    group.state = group.state.map(expand_states)
    try:
        group.state = group.state.fillna(group.state.mode()[0])
        return group
    except:
        return group


def abbrev_to_full(state):
    """
    Converts state abbreviations to full state names using a predefined mapping.
    
    Args:
        state: State name or abbreviation to convert
        
    Returns:
        Full state name if abbreviation exists in mapping, otherwise returns original
    """
    if state is not None:
        no_space = state.replace(" ", "")
        if len(no_space) < 3:
            state = no_space
    if state in utils.state_abbreviation_to_name:
        return utils.state_abbreviation_to_name[state]
    return state

print("Location processing functions defined!")

Location processing functions defined!


## Clean Location Data

In [13]:
print("Processing location data...")

# Create a copy for cleaning
df_clean = df.copy()

# Clean location strings
df_clean['loc'] = df_clean['loc'].str.replace(".", "")
df_clean['loc'] = df_clean['loc'].str.lower()

# Split into city and state
df_clean[['city', 'state']] = df_clean['loc'].str.split(',', n=1, expand=True)

# Clean state names
df_clean.state = df_clean.state.str.strip()
df_clean.state = df_clean.state.apply(abbrev_to_full)

# Clean city names
df_clean.city = df_clean.city.str.strip()

# Standardize cities
df_clean = df_clean.groupby(['city'], group_keys=False).apply(combine_cities)

# Fill in missing states
if hasattr(utils, 'missing_cities'):
    df_clean.state = df_clean.apply(
        lambda x: utils.missing_cities[x.city] 
        if x.state is None and x.city in utils.missing_cities 
        else x.state, 
        axis=1
    )

# Filter out non-US locations
original_size = len(df_clean)
df_clean = df_clean[~df_clean.state.isin(['mexico'])]
df_clean = df_clean[df_clean.state.notna()]
df_clean = df_clean.reset_index(drop=True)

# Map to regions
df_clean['region'] = df_clean.state.map(utils.region_mapping)

print(f"\nOriginal size: {original_size:,}")
print(f"After filtering: {len(df_clean):,}")
print(f"Filtered out: {original_size - len(df_clean):,} rows")
print(f"\nRegion distribution:")
print(df_clean['region'].value_counts())

Processing location data...


/tmp/ipykernel_3343318/273103754.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_clean = df_clean.groupby(['city'], group_keys=False).apply(combine_cities)



Original size: 412,225
After filtering: 403,822
Filtered out: 8,403 rows

Region distribution:
region
northeast    276698
southeast     48436
midwest       42520
west          34124
southwest      2032
Name: count, dtype: int64


## Save Processed Data

In [17]:
# Check if output file already exists
if os.path.exists(OUTPUT_FILE):
    print(f"WARNING: {OUTPUT_FILE} already exists!")
    response = input("Do you want to overwrite it? (yes/no): ")
    if response.lower() != 'yes':
        print("Aborting save. Please change OUTPUT_FILE in the configuration cell.")
    else:
        df_clean.to_json(OUTPUT_FILE, orient='records', lines=True, compression='gzip')
        print(f"\nSaved processed data to: {OUTPUT_FILE}")
        print(f"File size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.2f} MB")
else:
    df_clean.to_json(OUTPUT_FILE, orient='records', lines=True, compression='gzip')
    print(f"\nSaved processed data to: {OUTPUT_FILE}")
    print(f"File size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.2f} MB")


Saved processed data to: /data/mourad/narratives/proquest/processed_data_2010-2025_updated.jsonl.gz
File size: 38.66 MB


## Summary Statistics

In [18]:
print("=" * 60)
print("PROCESSING COMPLETE")
print("=" * 60)
print(f"\nInput file:  {INPUT_FILE}")
print(f"Output file: {OUTPUT_FILE}")
print(f"\nFinal dataset:")
if 'overlap_rows' in globals():
    print(f"  - Overlapping sentences removed: {overlap_rows:,}")
print(f"  - Total articles: {len(df_clean):,}")
print(f"  - Date range: {df_clean.year.min()}-{df_clean.year.max()}")
print(f"  - States: {df_clean.state.nunique()}")
print(f"  - Cities: {df_clean.city.nunique()}")
print(f"  - Regions: {df_clean.region.nunique()}")
print(f"\nColumns in output: {df_clean.columns.tolist()}")
print("\nNext step: Copy this file to the inference directory")
print("See cell below for command")

PROCESSING COMPLETE

Input file:  /data/mourad/narratives/proquest/raw_from_TDM/processed-data-2010-2025_updated.csv.gz
Output file: /data/mourad/narratives/proquest/processed_data_2010-2025_updated.jsonl.gz

Final dataset:
  - Overlapping sentences removed: 1,308,961
  - Total articles: 403,822
  - Date range: 2010-2025
  - States: 52
  - Cities: 303
  - Regions: 5

Columns in output: ['file_id', 'year_month', 'title', 'loc', 'text', 'year', 'month', 'city', 'state', 'region']

Next step: Copy this file to the inference directory
See cell below for command


## Next Steps: Prepare for Inference

Run this command to copy the file to the inference directory:

```bash
# Create directory
mkdir -p /net/projects2/chai-lab/mourad/narratives-data/filtered_sentences_for_prediction/proquest_2010-2025_updated/

# Copy file
cp /data/mourad/narratives/proquest/processed_data_2010-2025_updated.jsonl.gz \
   /net/projects2/chai-lab/mourad/narratives-data/filtered_sentences_for_prediction/proquest_2010-2025_updated/all_filtered.jsonl.gz
```

Then update `predict_json.py` to add the new dataset configuration.